# Lab 08-04 · Combinational Building Blocks

Companion to chapter [08-04](../../chapters/08-04-combinational-building-blocks.md). Handout: [README.md](README.md).

Blocks are checked exhaustively against their specifications, and timing is simulated with the chapter's unit-delay model: every gate output at time $t$ is the gate's function of its inputs at time $t - 1$. NumPy lets us run hundreds of thousands of input transitions at once. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(804)

## Part 1 · Blocks against their specifications

**🔮 Predict:** of the 256 inputs of an 8:3 priority encoder, how many produce each output code?

In [2]:
def mux(sel, data):
    return data[sel]


def decoder(a, n):
    return [int(a == i) for i in range(2 ** n)]


def prio8(d):
    # equations of Problem 08-04.5
    D = [(d >> k) & 1 for k in range(8)]
    N = lambda x: 1 - x
    y2 = D[7] | D[6] | D[5] | D[4]
    y1 = D[7] | D[6] | (N(D[5]) & N(D[4]) & (D[3] | D[2]))
    y0 = D[7] | (N(D[6]) & D[5]) | (N(D[6]) & N(D[4]) & D[3]) | (N(D[6]) & N(D[4]) & N(D[2]) & D[1])
    return 4 * y2 + 2 * y1 + y0, int(d != 0)


def highest(d):
    return d.bit_length() - 1 if d else None


counts = {}
for d in range(256):
    y, valid = prio8(d)
    assert valid == (d != 0) and (y == highest(d) if d else y == 0)
    counts[y if valid else "none"] = counts.get(y if valid else "none", 0) + 1
print("priority encoder counts:", counts)

for n in (2, 3, 4):
    assert all(sum(decoder(a, n)) == 1 and decoder(a, n)[a] == 1 for a in range(2 ** n))
f = {1, 2, 6, 7}   # Example 08-04.1 with data inputs C, C', 0, 1
assert all(mux(2 * A + B, [C, 1 - C, 0, 1]) == int(4 * A + 2 * B + C in f) for A in (0, 1) for B in (0, 1) for C in (0, 1))
print("decoders and the multiplexer implementation of Example 08-04.1 check out")

priority encoder counts: {'none': 1, 0: 1, 1: 2, 2: 4, 3: 8, 4: 16, 5: 32, 6: 64, 7: 128}
decoders and the multiplexer implementation of Example 08-04.1 check out


## Part 2 · Settling time and glitches of a 4-bit ripple adder (the chapter's guided exercise)

**🔮 Predict:** the longest settling time of any output over every input change, and the share of output transitions that are glitches.

In [3]:
def rca_gates(n):
    g = []
    for i in range(n):
        g += [(f"p{i}", "xor", f"a{i}", f"b{i}"), (f"g{i}", "and", f"a{i}", f"b{i}"),
              (f"s{i}", "xor", f"p{i}", f"c{i}"), (f"q{i}", "and", f"p{i}", f"c{i}"),
              (f"c{i + 1}", "or", f"g{i}", f"q{i}")]
    return g


F = {"xor": np.bitwise_xor, "and": np.bitwise_and, "or": np.bitwise_or}


def simulate(n, old, new):
    # old, new: integer arrays packing a (n bits), b (n bits), c0 (1 bit). Returns per-output statistics.
    gates = rca_gates(n)
    names = [f"a{i}" for i in range(n)] + [f"b{i}" for i in range(n)] + ["c0"]
    unpack = lambda x: {nm: ((x >> k) & 1).astype(np.uint8) for k, nm in enumerate(names)}
    st = unpack(old)
    for _ in range(len(gates)):                      # settle on the old inputs
        for o, op, x, y in gates:
            st[o] = F[op](st[x], st[y])
    start = dict(st)
    cur = {**st, **unpack(new)}
    outs = [f"s{i}" for i in range(n)] + [f"c{n}"]
    trans = {o: np.zeros(len(old), np.int32) for o in outs}
    last = {o: np.zeros(len(old), np.int32) for o in outs}
    for t in range(1, 2 * n + 4):
        nxt = dict(cur)
        for o, op, x, y in gates:
            nxt[o] = F[op](cur[x], cur[y])
        for o in outs:
            ch = nxt[o] != cur[o]
            trans[o] += ch
            last[o] = np.where(ch, t, last[o])
        cur = nxt
    need = {o: (start[o] != cur[o]).astype(np.int32) for o in outs}
    return outs, trans, last, need, cur


n = 4
N9 = 2 ** (2 * n + 1)
old = np.repeat(np.arange(N9), N9)
new = np.tile(np.arange(N9), N9)
outs, trans, last, need, cur = simulate(n, old, new)
total = sum(trans[o] for o in outs)
needed = sum(need[o] for o in outs)
print(f"{len(old)} transitions")
print("longest settling, sum bits:", max(last[o].max() for o in outs[:-1]), " carry out:", last["c4"].max())
print(f"mean transitions {total.mean():.3f}, needed {needed.mean():.3f}, glitch share {100 * (1 - needed.sum() / total.sum()):.1f}%")
assert max(last[o].max() for o in outs[:-1]) == 8 and last["c4"].max() == 9

262144 transitions
longest settling, sum bits: 8  carry out: 9
mean transitions 5.914, needed 2.500, glitch share 57.7%


In [4]:
worst = np.argmax(last["c4"])
a_old, b_old, c_old = old[worst] & 15, (old[worst] >> 4) & 15, old[worst] >> 8
a_new, b_new, c_new = new[worst] & 15, (new[worst] >> 4) & 15, new[worst] >> 8
print(f"one worst case: A,B,Cin = {a_old:04b},{b_old:04b},{c_old} -> {a_new:04b},{b_new:04b},{c_new}")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
bins = np.arange(0, 11) - 0.5
ax.hist(np.max(np.stack([last[o] for o in outs]), axis=0), bins=bins, color="#0072B2", rwidth=0.85)
ax.set_xlabel("settling time of the slowest output (units)")
ax.set_ylabel("input transitions")
ax.set_title("4-bit ripple-carry adder, all 262 144 transitions")
plt.show()

one worst case: A,B,Cin = 0000,0000,0 -> 1111,0000,1


Most transitions settle well before the worst case: the long carry ripple needs a specific pattern (propagate in every column). But a clocked design must wait for the worst case every cycle.

## Part 3 · Glitches against width

**🔮 Predict:** as the adder widens, do glitch transitions grow faster or slower than needed transitions?

In [5]:
for n in (4, 8, 16):
    K = 20000
    old = rng.integers(0, 2 ** (2 * n + 1), K)
    new = rng.integers(0, 2 ** (2 * n + 1), K)
    outs, trans, last, need, cur = simulate(n, old, new)
    total = sum(trans[o] for o in outs).mean()
    needed = sum(need[o] for o in outs).mean()
    print(f"n = {n:2d}: needed {needed:5.2f}, glitches {total - needed:6.2f} per addition "
          f"(ratio {(total - needed) / needed:.2f}), worst settling {max(l.max() for l in last.values())} units")

n =  4: needed  2.50, glitches   3.39 per addition (ratio 1.36), worst settling 9 units
n =  8: needed  4.52, glitches   7.42 per addition (ratio 1.64), worst settling 17 units
n = 16: needed  8.50, glitches  15.34 per addition (ratio 1.80), worst settling 32 units


Needed transitions grow in proportion to $n$ (about half the outputs change), while glitches grow faster, because longer carry chains give each sum bit more chances to flip and flip back.

## Part 4 · A static hazard, step by step

Example 08-04.7: $Y = \overline{S}D_0 + SD_1$ with $D_0 = D_1 = 1$ while $S$ falls.

In [6]:
def trace(gates, start, change, steps=6):
    ops = {"not": lambda a: 1 - a, "and": lambda *x: int(all(x)), "or": lambda *x: int(any(x))}
    st = dict(start)
    for _ in range(len(gates)):
        for o, op, ins in gates:
            st[o] = ops[op](*(st[i] for i in ins))
    cur = {**st, **change}
    rows = [dict(cur)]
    for _ in range(steps):
        nxt = dict(cur)
        for o, op, ins in gates:
            nxt[o] = ops[op](*(cur[i] for i in ins))
        cur = nxt
        rows.append(dict(cur))
    return rows


mux_min = [("sn", "not", ["s"]), ("t0", "and", ["sn", "d0"]), ("t1", "and", ["s", "d1"]), ("y", "or", ["t0", "t1"])]
mux_fix = mux_min[:-1] + [("t2", "and", ["d0", "d1"]), ("y", "or", ["t0", "t1", "t2"])]
for label, g in (("minimal", mux_min), ("with consensus D0*D1", mux_fix)):
    rows = trace(g, {"s": 1, "d0": 1, "d1": 1}, {"s": 0})
    print(label)
    for t, r in enumerate(rows):
        print(f"  t={t}: s={r['s']} sn={r['sn']} t0={r['t0']} t1={r['t1']} y={r['y']}")
ys = [r["y"] for r in trace(mux_min, {"s": 1, "d0": 1, "d1": 1}, {"s": 0})]
assert ys[2] == 0 and ys[1] == 1 and ys[3] == 1
assert all(r["y"] == 1 for r in trace(mux_fix, {"s": 1, "d0": 1, "d1": 1}, {"s": 0}))

minimal
  t=0: s=0 sn=0 t0=0 t1=1 y=1
  t=1: s=0 sn=1 t0=0 t1=0 y=1
  t=2: s=0 sn=1 t0=1 t1=0 y=0
  t=3: s=0 sn=1 t0=1 t1=0 y=1
  t=4: s=0 sn=1 t0=1 t1=0 y=1
  t=5: s=0 sn=1 t0=1 t1=0 y=1
  t=6: s=0 sn=1 t0=1 t1=0 y=1
with consensus D0*D1
  t=0: s=0 sn=0 t0=0 t1=1 y=1
  t=1: s=0 sn=1 t0=0 t1=0 y=1
  t=2: s=0 sn=1 t0=1 t1=0 y=1
  t=3: s=0 sn=1 t0=1 t1=0 y=1
  t=4: s=0 sn=1 t0=1 t1=0 y=1
  t=5: s=0 sn=1 t0=1 t1=0 y=1
  t=6: s=0 sn=1 t0=1 t1=0 y=1


The minimal multiplexer's output is 0 at $t = 2$ only: a 1-unit glitch equal to the inverter's delay. The consensus term $D_0D_1$ holds the output at 1 throughout. The Verilog testbench `hazard_tb.v` shows the same thing with 1 ns gate delays.

## What to take away

1. A multiplexer is a universal function generator; a decoder is a minterm generator.
2. A ripple adder's delay is linear in its width, and its worst case needs a carry that travels the whole way.
3. Most of a ripple adder's switching, in this model, is glitches: wasted energy, harmless to correctness only because flip-flops sample after settling.
4. Consensus terms remove static hazards where glitches matter.